# Flask + Jinja + HTML: les templates

On sait maintenant:
- comment créer une application Flask 
- comment créer des routes, et des routes à paramètres
- comment afficher du texte dans le navigateur

**Mais**: pour le moment, notre site est MOCHE, et, plus grave, il n'utilise pas du tout de HTML, qui est quand même la base du web. On est assez loin d'avoir fait le nouveau Gallica. Aujourd'hui, on a donc apprendre à **générer du HTML en Python dans une appli Flask, via la librairie Jinja**.

---

# Flask + HTML

Pour le moment, nos fonctions ne renvoient que du texte:

In [2]:
from flask import Flask

app = Flask("Ma première appli")

@app.route("/")
def index():
    return "Hello world !"

Pour afficher du HTML, on peut **créer des vues qui retournent du HTML au lieu du texte**, mais ça va vite devenir douloureux à écrire et très difficile à maintenir. Donc, on ne verra qu'une fois comment faire, dans `./apps/s2/html/main.py`:

In [3]:
app = Flask("templates HTML")

@app.route("/")
def index():
    app_name = "Catalogue Richelieu"
    html = f"""
        <html>
            <head>
                <title>{app_name}</title>
            </head>
            <body>
                <h1>Bienvenue sur le {app_name} !</h1>
            </body>
        </html>
    """
    return html

Pour monter une vraie appli, on va avoir besoin d'un système beaucoup plus complet pour générer dynamiquement du HTML à partir de Python, et c'est ici que les templates rentrent en jeu.

---

# Templates Jinja

## Jinja ?

**[Jinja](https://jinja.palletsprojects.com/en/stable/) est un "moteur de templates"** (*template engine*): une librairie qui, à partir de fichiers templates, génère dynamiquement du HTML adapté à vos données et votre code Python. **Parmi les opérations possibles**:
- remplacer des variables Python par leurs valeurs
- boucler sur des variables Python pour créer des listes
- gérer des structures conditionnelles (`if`/`else`), pour n'afficher du contenu que si certaines conditions sont réunies
- combiner ensemble plusieurs templates
- et plein d'autres choses !

Jinja est une librairie développée par [Pallets projects](https://palletsprojects.com/), qui sont les créateurs de Flask. L'intégration Flask/Jinja est donc très transparente. À noter:
- Jinja peut être utilisé sans Flask
- Jinja peut être utilisé pour générer n'importe quel type de fichier texte: HTML, XML, Latex...

## Une appli avec templates

Une première application utilisant Jinja se trouve dans `./apps/s2/jinja_base/main.py`. Lançons la.

### Code

Pour que notre vue retourne une template Jinja plutôt qu'une chaîne de caractère, il faut juste utiliser la fonction `render_template`:

In [6]:
from flask import render_template

app_name = "Catalogue Richelieu"

app = Flask(app_name)

@app.route("/")
def index():
    return render_template("homepage.html", app_name=app_name)

### Syntaxe

#### Côté python

Pour utiliser `render_template`, 
- on **importe la fonction**
- on **fait une vue qui retourne le résultat de `render_template`**. `render_template` prend pour arguments:
    - en 1er argument **le nom d'une template Jinja**: `homepage.html`
    - ensuite, **les valeurs à rendre accessibles dans le template**: `app_name=app_name`. La syntaxe est: `nom_de_variable_dans_la_template=nom_de_variable_en_python`.

#### Côté Jinja

Voici `homepage.html`:

```html
<html>
    <head>
        <title>{{app_name}}</title>
    </head>
    <body>
        <h1>Bienvenue sur le {{app_name}} !</h1>
    </body>
</html>
```

On voit que c'est un fichier HTML, **mais**, qui contient `{{app_name}}`: 
- les noms de variables Python sont **encadrés de doubles accolades (`{{}}`)**
- `render_template` **remplacera le nom de variable** par sa valeur: c'est de *l'interpolation de variables*

## Bonnes pratiques: structurer le code d'une application Flask

Jusqu'à maintenant, toutes nos applis étaient contenues dans un seul fichier. Maintenant, on arrive à **une appli organisée en module** (un dossier avec plusieurs sous-dossiers). Voilà comment l'appli est organisée, et voilà comment toutes nos applis seront plus où moins organisées pour le reste de nos cours:

```txt
├── main.py ................: fichier qui lance l'appli
└──app ....................: dossier contenant notre application
   ├── app.py .............: fichier qui définit notre `app` Flask 
   ├── __init__.py
   ├── routes .............: dossier qui contient toutes nos routes Flask
   │   ├── generic.py .....: pour le moment, toutes nos routes seront contenues dans ce fichier 
   │   └── __init__.py
   ├── templates ..........: dossier contenant nos templates HTML
   │   └── homepage.html ..: la template affichée juste au dessus
   └── utils ..............: code utilitaire
       └── constants.py ...: toutes les constantes
```

Regardons dans le détail, en commençant par les petits blocs.

### `app/templates/`

Ce dossier contient **toutes nos templates Jinja HTML**.

### `app/utils/` et `constants.py`

**Le dossier `utils/` stocke les "utilitaires"**, c'est à dire les petites fonctions qui n'ont pas vraiment d'autres endroits où aller et qui sont utilisées partout dans l'appli (par ex.: fonctions qui gèrent la lecture/écriture de fichiers). Idéalement, `utils/` ne doit pas contenir des éléments "sensibles" (comme des opérations sur base de données).

On y met `constants.py` qui contient toutes les *constantes* (variables définies au lancement de l'appli qui seront pas modifiées ensuite). Ces constantes sont nos chemins vers des fichiers/dossiers importants:

```py
# chemin absolu vers notre dossier de templates (app/templates/)
DIR_TEMPLATES = DIR_APP / "templates" 
```

### `app/routes/`et `generic.py`

**Le dossier `routes/` va contenir toutes les routes** de notre appli. Pour l'instant, toutes nos routes sont dans un fichier (`generic.py`), mais si notre appli grandit c'est une bonne idée de les scinder en plusieurs fichiers.

Voilà le contenu de `generic.py`:

```py
from flask import render_template

# note: app.app = app/app.py => on importe la variable app du fichier app/app.py
from app.app import app

@app.route("/")
def index():
    app_name = "Catalogue Richelieu"
    return render_template("homepage.html", app_name=app_name)
```

On voit que:
- **on importe `app`**, définie dans `app/app.py` pour pouvoir utiliser `@app.route`
- pour le reste, **le code est inchangé**

### `app/app.py`

**C'est le ficher qui définit notre appli Flask**. Voilà son contenu:

```py
from flask import Flask

from app.utils.constants import DIR_TEMPLATES

app = Flask(
    "Jinja base",
    template_folder=DIR_TEMPLATES
)

from app.routes import generic
```

À noter:
- `template_folder=DIR_TEMPLATES` permet d'indiquer à Flask **où chercher les templates Jinja**: dans notre dossier `templates/`. 
    - => **tous les chemins vers des templates dans `render_template`** seront définis relativement au dossier `templates/`.
    - à noter que dans certaines configurations, `template_folder` est optionnel.
- `from app.routes import generic` permet **d'importer les routes** pour qu'elles soient utilisables.
    - d'habitude, les imports sont faits au début du fichier.
    - ici, c'est important **d'importer routes.generic à la fin du fichier pour éviter un import circulaire**. sinon, 
        ```txt
        ┌─────────────────────────────────────────────────────────────┐
        │                                                             │
        │   1. `app/routes/generic.py`                                │
        │      exécute  →  from app import app                        │
        │   2. python doit résoudre le module `app`                   │
        │      →  il commence à exécuter `app/app.py`                 │
        │   3. or `app/app.py` exécute lui-même:                      │
        │      →  `import app.routes.generic`                         │
        │   4. python revient donc dans `app/routes/generic.py`,      │
        │      → python doit résoudre le module `app`                 │
        │                                                             │
        │   → import circulaire: boucle infinie !                     │
        │                                                             │
        └─────────────────────────────────────────────────────────────┘
        ``` 

### `main.py`

**C'est le fichier qui lance notre appli Flask** et le fichier qu'on éxécute avec Python:

```py
from app.app import app

if __name__ == "__main__":
    app.run(debug=True)
```


---
---

 programme des hostilités:
- HTML basique
- HTML avec interpolation de variables, et pourquoi c relou
- templates Jinja:
    - `render_template`
    - syntaxe: interpolation de variables
    - synxate: for
    - syntaxe: if/then/else
    - syntaxe: include/extends (où mettre ça ?)
    - syntaxe: les filtres et la manipulation de variables
    - un brin de CSS
- la gestion d'erreurs